## **Training of MODEL3**

In [1]:
import pandas as pd
import numpy as np
from keras.src.ops import dtype
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import joblib
import os

In [2]:
TARGET_VARIABLE = 'Time to Depletion'

In [3]:
data = pd.read_csv("../new_code/DATASET.csv")

In [4]:
data = data.drop('Remaining Capacity', axis = 1)

In [5]:
data.head()

,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Time to Depletion,type,capacity,charged
0,9.36,11.84,0.156000,0.156000,110.8224,10324.615385,b2,88.81,27.0
1,9.34,11.84,0.155667,0.311667,110.5856,10286.723769,b2,88.81,27.0
2,9.34,11.83,0.155667,0.467333,110.4922,10226.723769,b2,88.81,27.0
3,7.14,11.88,0.119000,0.586333,84.8232,13317.815126,b2,88.81,27.0
4,7.13,11.88,0.118833,0.705167,84.7044,13276.493689,b2,88.81,27.0


In [6]:
Y = data[TARGET_VARIABLE]
X = data.drop(TARGET_VARIABLE, axis=1)

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical Featrures are : ", numerical_features)
print("Categorical Featrures are : ", categorical_features)

Numerical Featrures are :  ['Current', 'Voltage', 'Ah Out', 'Cumulative Actual Disch Ah', 'Power', 'capacity', 'charged']
Categorical Featrures are :  ['type']


In [7]:
print("NaN locations:")
for column in data.columns:
    if data[column].isna().any():
        print(f"\n{column}:")
        print(data[data[column].isna()].index)


NaN locations:


In [8]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15, random_state=42)

In [9]:
numerical_transformer = Pipeline(steps=[
    ('pass',
     'passthrough')
])
categorical_transformer = Pipeline(steps=[
    ('onehot',
     OneHotEncoder(handle_unknown='ignore',
                   sparse_output=False))
])

In [10]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
]
    ,remainder='passthrough')

In [11]:
rf_model = RandomForestRegressor(
    random_state=42,
    bootstrap=True,
    criterion='absolute_error',
    n_jobs=-1,
    n_estimators=150,
    max_depth=14,
)

In [12]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', rf_model)
])

In [13]:
best_match = pipeline.fit(X_train, Y_train)

In [14]:
Y_pred = best_match.predict(X_test)

In [15]:
mae = mean_absolute_error(Y_test, Y_pred)

print(f"Mean Absolute Error is : {mae:.2f}")

Mean Absolute Error is : 285.40
